# 01 — HOG + SVM starter
Setup and data loading work now. Model sections are intentionally TODOs: no HOG extraction, training, tuned hyperparameters, or results are supplied.

## 1. Setup

### Environment and dataset configuration
**The one dataset path you may need to change is `DRIVE_DATA_ROOT` below.** It is an example, not a known shared-folder location. Point to the folder directly containing `Train.csv`, `Test.csv`, `Train/`, and `Test/`. If the folder is under "Shared with me", add a shortcut under My Drive first.

Colab defaults to Drive; local runs default to `Dataset/raw/`. Set `USE_DRIVE = False` to use repository data in Colab. Run cells in order. No API keys are needed.

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB
# CHANGE THIS ONE DATASET PATH to your own shared Drive folder/shortcut:
DRIVE_DATA_ROOT = "/content/drive/MyDrive/CP467_Traffic_Signs/data/raw"
print("Running in Colab:", IN_COLAB)
print("Python:", sys.version)

### Mount Drive and locate project code
Opening a GitHub notebook in Colab does not clone its Python files. This cell finds a local project checkout or clones it once in Colab. For a code checkout stored on Drive, change the working directory to that checkout before this cell (see README).

In [ ]:
import os
import subprocess

if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None and IN_COLAB:
    PROJECT_ROOT = Path("/content/CP467GroupProject")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Davy-Chu/CP467GroupProject.git", str(PROJECT_ROOT)],
            check=True,
        )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Open from the project checkout or change directory to its root first.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

### Dependencies
Colab often includes these libraries. Leave installation off first; if an import fails, set `INSTALL_DEPENDENCIES = True`, run this cell, restart the runtime if requested, then rerun from the top with installation off. Avoid replacing a working Colab GPU environment unnecessarily.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import skimage
import sklearn
import torch
import torchvision
import seaborn

for library in (cv2, np, pd, matplotlib, skimage, sklearn, torch, torchvision, seaborn):
    print(f"{library.__name__}: {library.__version__}")

In [ ]:
from src.config import (
    NUM_CLASSES, IMAGE_SIZE, RANDOM_SEED, REPO_DATA_ROOT,
    RESULTS_DIR, MODELS_DIR, CONDITIONS, DEGRADATION_PARAMS,
    get_data_root, seed_everything,
)
from src.data_utils import (
    validate_dataset, load_metadata, load_image,
    make_train_validation_split, split_fingerprint,
)
from src.degradations import (
    apply_condition, reduce_brightness, add_noise, apply_motion_blur,
    equalize_histogram, gaussian_filter,
)

seed_everything()
DATA_ROOT = get_data_root(USE_DRIVE, DRIVE_DATA_ROOT)
print("Selected data root:", DATA_ROOT)
print("Repository fallback:", REPO_DATA_ROOT)
print("Shared conditions:", DEGRADATION_PARAMS)

## 2. Load Dataset

Validation requires all 43 classes, the full 39,209/12,630 labeled image counts, and every referenced file. It fails if anything required is missing. The default decodes one image per class in each split; use `check_images="all"` for a slower full corruption scan on Drive. Do not disable the count checks to work around an unfinished upload.

In [ ]:
# This must pass before either team begins model development.
training_metadata, test_metadata = validate_dataset(DATA_ROOT, check_images="sample")
train_metadata, validation_metadata = make_train_validation_split(training_metadata)
SPLIT_ID = split_fingerprint(train_metadata, validation_metadata)
print(f"Training subset: {len(train_metadata):,}")
print(f"Validation subset: {len(validation_metadata):,}")
print(f"Official test (final evaluation only): {len(test_metadata):,}")
print("Split ID (must match across teammates):", SPLIT_ID)

## 3. Preprocessing
Apply any experimental condition to the common RGB image before grayscale conversion. Fit learned preprocessing on the training subset only.

In [ ]:
# TODO: Convert the common resized RGB images to grayscale; document normalization.
# Compare optional equalize_histogram / gaussian_filter separately, with parameters logged.

## 4. HOG Feature Extraction

In [ ]:
# TODO: Choose and document HOG settings. Extract features for the shared subsets.
# Use skimage.feature.hog; tune settings on validation only.

## 5. Train SVM

In [ ]:
# TODO: Build the SVM pipeline. Fit scaling (if used) and SVM only on train_metadata.
# Choose model settings with validation; do not inspect test scores for tuning.

## 6. Validation

In [ ]:
# TODO: Evaluate on validation_metadata using accuracy and macro F1 over all 43 classes.
# Fix model and preprocessing choices before final test evaluation.

## 7. Clean Test Evaluation

In [ ]:
# TODO: Evaluate the frozen pipeline on test_metadata with condition='clean'.
# Use sklearn.metrics.accuracy_score and f1_score(..., average='macro',
# labels=list(range(NUM_CLASSES)), zero_division=0).

## 8. Degraded Test Evaluation

In [ ]:
# TODO: Loop over CONDITIONS. Use apply_condition(image, condition, image_id=row.Path).
# Keep the same test rows, resizing, parameters, and optional preprocessing as CNN.
# Do not save duplicate degraded datasets or tune on degraded test scores.

## 9. Timing

In [ ]:
# TODO: Use time.perf_counter for training time and mean inference milliseconds/image.
# Record hardware, batch size, warmup, and whether preprocessing is included.
# Agree on the same timing boundaries with the CNN team.

## 10. Confusion Matrix

In [ ]:
# TODO: Compute a 43x43 matrix with labels=list(range(NUM_CLASSES)).
# Save labeled figures under RESULTS_DIR / 'confusion_matrices'.

## 11. Save Results
The metrics file contains a header only. See the README for metric definitions and artifact guidance.

In [ ]:
# TODO: Append real measurements to RESULTS_DIR / 'metrics.csv' using its header.
# Record preprocessing, RANDOM_SEED, and SPLIT_ID; save plots under results/plots/.
# Save a reasonably sized final pipeline under MODELS_DIR / 'hog_svm.pkl'.
# Keep temporary checkpoints out of Git.